# 00. Revisión de datos

Revisión inicial de las bases originales. No se eliminan importes negativos ni filas de forma automatica.

**Pasos:**

- Importar las bases de emisión y siniestros

- Revisar dimensiones, tipos de variables, nulos y duplicados (los duplicados no necesariamente son errores de registro, no eliminar)

- Identificar ceros y valores negativos (Pregutar que hacer con los valores negativos y ceros)

- Normalizar categorías de embarcación y uso (Hay tipos de categorias las cuales son la misma pero por un caracter diferente son consideradas diferentes)

- Documentar las decisiones de limpieza

## Importación e inspección

In [1]:
# ------------------------------------------
# 1. Importar librerías
# ------------------------------------------

from pathlib import Path                   # Trabajar con rutas de archivos y carpetas
import sys                                 # Permitir importar funciones de otras carpetas
import pandas as pd                        # Manipulación de datos
import numpy as np                         # Operaciones numéricas
from IPython.display import display, HTML  # Mostrar tablas HTML en el notebook

# ------------------------------------------
# 2. Localizar la carpeta principal del proyecto
# ------------------------------------------

# Buscar la carpeta que contiene funciones_reservas.py,
# comenzando desde la carpeta actual y revisando sus carpetas superiores.
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "funciones_reservas.py").exists()),
    None
)

# Si no se encuentra el archivo, detener la ejecución.
if ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto con funciones_reservas.py."
    )

# ------------------------------------------
# 3. Configurar la importación de funciones
# ------------------------------------------

# Agregar la carpeta del proyecto a las rutas donde Python
# busca archivos cuando utilizamos import.
sys.path.insert(0, str(ROOT))


# ------------------------------------------
# 4. Definir las rutas de trabajo
# ------------------------------------------

RAW = ROOT / "data" / "raw"             # Bases de datos originales
PROCESSED = ROOT / "data" / "processed" # Bases limpias y ajustadas


In [2]:
# ------------------------------------------
# 1.1.1 Importación de las bases de datos
# ------------------------------------------

# Importar las funciones de calidad y limpieza desde nuestro archivo de funciones
from funciones_reservas import normalizar_categorias, resumen_calidad

# Definir la ubicación del Excel original
archivo = RAW / "2026 BASE RESERVAS.xlsx"

# Cargar las hojas de emisión y siniestralidad como DataFrames
df1 = pd.read_excel(archivo, sheet_name="EMISION")
df2 = pd.read_excel(archivo, sheet_name="SINIESTRALIDAD")

# Mostrar las dimensiones de ambas bases (filas, columnas)
print("===== Dimensiones de los DataFrames =====")
print(f"Emisión: {df1.shape}; Siniestros: {df2.shape}")

===== Dimensiones de los DataFrames =====
Emisión: (21353, 7); Siniestros: (2763, 10)


In [3]:
# ------------------------------------------
# 1.1.2 Resumen de calidad de los datos
# ------------------------------------------

tablas = []
for nombre, df in [("Emisión", df1), ("Siniestros", df2)]:
    resumen = resumen_calidad(df)
    tabla = f"""
    <div style="flex:1; min-width:0;">
        <h3>{nombre}</h3>
        {resumen.to_html()}
    </div>
    """
    tablas.append(tabla)
display(HTML('<div style="display:flex; gap:30px; align-items:flex-start;">' + ''.join(tablas) + '</div>'))


,tipo,nulos,unicos,ceros,negativos
año,int64,0,11,0,0
tipo_embarcacion,object,0,42,—,—
uso,object,0,12,—,—
anio_construccion,int64,0,90,0,0
prima_emitida,int64,0,18755,0,1126
prima_retenida,int64,0,15882,2009,938
comision_directa,int64,0,11827,2226,857
,tipo,nulos,unicos,ceros,negativos
año,int64,0,11,0,0
tipo_embarcacion,object,0,33,—,—


In [4]:
# ------------------------------------------
# 1.1.3 Limpieza de las bases de datos
# ------------------------------------------

emision = df1.copy().dropna()
siniestros = df2.copy().dropna()
for nombre, df in [("emision", emision), ("siniestros", siniestros)]:
    columnas = [col for col in ["tipo_embarcacion", "uso"] if col in df]
    if columnas:
        df_actual = normalizar_categorias(df, columnas, minusculas=True)    # Normalizar categorías de las columnas especificadas
        df.loc[:, columnas] = df_actual[columnas]
    if "anio_construccion" in df:
        df["anio_construccion"] = pd.to_numeric(df["anio_construccion"], errors="coerce").astype("Int64")


In [7]:
# ----------------------------------------------
# 1.1.4 Resumen de calidad de los datos limpios
# ----------------------------------------------

tablas = []

for nombre, df in [("Emisión", emision), ("Siniestros", siniestros)]:
    resumen = resumen_calidad(df)

    tablas.append(f"""
    <div style="flex: 1; min-width: 0;">
        <h3>{nombre}</h3>
        {resumen[["unicos", "ceros", "negativos"]].to_html()}
    </div>
    """)

display(HTML(
    '<div style="display: flex; gap: 30px; align-items: flex-start;">'
    + "".join(tablas) +
    "</div>"
))


,unicos,ceros,negativos
año,11,0,0
tipo_embarcacion,32,—,—
uso,11,—,—
anio_construccion,90,0,0
prima_emitida,18755,0,1126
prima_retenida,15882,2009,938
comision_directa,11827,2226,857
,unicos,ceros,negativos
año,11,0,0
tipo_embarcacion,25,—,—


### Presencia de valores negativos

Los importes negativos podrían corresponder a cancelaciones de pólizas, devoluciones o ajustes contables. Lo correcto no seria eliminar estos registros negativos ya que pueden representar algo importante.

Al sumar las primas positivas y negativas obtenemos el monto neto, esto significa que nuestro analisis será en terminos netos monetarios ajustados

Existen tipos de embarcación con primas agregadas negativas en determinados años. 

No hay valores faltantes.

Un cero puede representar la ausencia de un pago.

## Hallazgos

Las decisiones sobre faltantes, signos negativos, categorias y ceros deben conservarse documentadas antes de calcular reservas.